In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import sys
sys.path.append('/media/dll-1/SSD 4TB/code/LEEDNet/')  # Replace with real path
from EEGDataLoader import EEGDataset
from models.LEEDNetModel import LEEDNetModel
import config
from train_eval import eval, train

model_saving_name = "/media/dll-1/SSD 4TB/EEG Datasets/experiment/nouman_conv_2/model/weights/conv_tt_02_tukl_5_aug.pt"

epochs = 30
batch_size = 64
hidden_size = 20
output_size = 3  # Binary classification
number_of_eeg_channels = 1
eval_batch = 1
threshold = 0.495
active_branches = [0,2]

In [ ]:
train_dataset = EEGDataset(config.train_data_csv, number_of_eeg_channels=number_of_eeg_channels)
val_dataset = EEGDataset(config.val_data_csv, number_of_eeg_channels=number_of_eeg_channels)
test_dataset = EEGDataset(config.test_data_csv, number_of_eeg_channels=number_of_eeg_channels)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=eval_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=eval_batch, shuffle=False)

In [ ]:
input = train_dataset.__getitem__(0)
input_x1 = input[1].shape
input_x2 = input[2].shape
input_x3 = input[3].shape
criterion = nn.CrossEntropyLoss()

loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

print(input_x1,input_x2,input_x3)
input_channels_list = [input_x1[1], input_x2[1], input_x3[1]]

model = LEEDNetModel(input_channels_list=input_channels_list, output_size=3)

main_output, aux_outputs = model([input[1], input[2], input[3]], active_branch_indices=active_branches)
print("Output shape:", main_output.shape) 

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001,betas=(0.9, 0.98), eps=1e-9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:57: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at ../aten/src/ATen/native/Copy.cpp:305.)
  coefficients_cmor = torch.tensor(coefficients_cmor, dtype=torch.float32)
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)


torch.Size([1, 1, 400]) torch.Size([1, 33, 14]) torch.Size([1, 30, 400])
Output shape: torch.Size([1, 3])


ConvParallelEEG1DModel(
  (branches): ModuleList(
    (0): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(in_features=32, out_features=3, bias=True)
    )
    (1): ConvBranch1D(
      (conv): Sequential(
        (0): Conv1d(33, 32, kernel_size=(3,), stride=(1,), padding=(1,))
        (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (4): Dropout(p=0.3, inplace=False)
      )
      (global_pool): AdaptiveAvgPool1d(output_size=1)
      (linear): Linear(

In [4]:
prev_eval_loss = 0
prev_eval_acc = 0


In [5]:
model = model.to(device)

prev_eval_loss, prev_eval_acc = eval(model, val_loader, loss_fn, 1,False,'', active_branches, device)
print("Loss:",prev_eval_loss, "Accuracy",prev_eval_acc)

Eval:   0%|          | 0/43343 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x704f348de160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 43343/43343 [02:16<00:00, 317.90it/s]

Correct classified: 5936.0 / 43343
Class-wise correct: Class 0 = 0.0, Class 1 = 87.0, Class 2 = 5849.0
Precision: 32.66  Recall: 33.53  F1: 8.33
Confusion Matrix:
 [[    0    16 22522]
 [    0    87 14869]
 [    0     0  5849]]
Loss: 1.1114476815155199 Accuracy 0.13695406409339456



/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [6]:
result = {"epoch":[],"train_loss":[],"train_acc":[],"eval_loss":[],"eval_acc":[],"test_acc":[],"test_loss":[]}
saved_epoch = 0

In [7]:
for epoch in range(epochs):  # loop over the dataset multiple times
    train_loss, train_acc = train(model, train_loader,optimizer, loss_fn, active_branches, device)
    print(f"Train Loss after epoch {epoch+2}: {train_loss} Train ACC after epoch {epoch}: {train_acc}")
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    print(f"VAL Loss after epoch {epoch}: {eval_loss} VAL ACC after epoch {epoch}: {eval_acc}")
    # test_loss, test_acc = eval(model, test_loader, loss_fn, eval_batch,False,'', active_branches, device)
    test_loss = eval_loss
    test_acc = eval_acc
    print(f"Test Loss after epoch {epoch}: {test_loss} Test Acc after epoch {epoch}: {test_acc}")

    scheduler.step(eval_loss)
    
    result['epoch'].append(epoch)
    
    result['train_loss'].append(train_loss)
    result['train_acc'].append(train_acc)
    result['eval_loss'].append(eval_loss)
    result['eval_acc'].append(eval_acc)
    result['test_loss'].append(test_loss)
    result['test_acc'].append(test_acc)

    if prev_eval_acc < eval_acc:
      prev_eval_acc = eval_acc
      saved_epoch = epoch
      torch.save(model.state_dict(), model_saving_name)
      print("WEIGHTS SAVED")

Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x704f348de160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Training: 100%|██████████| 4531/4531 [10:13<00:00,  7.39it/s]


Correct Classified: 205100.0 / 289972
Class-wise correct: Class 0 = 123001.0, Class 1 = 26489.0, Class 2 = 55610.0
Train Loss after epoch 2: 1.7115727017136815 Train ACC after epoch 0: 0.7073096712786062


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x704f348de160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 74774/74774 [03:44<00:00, 332.74it/s]


Correct classified: 55245.0 / 74774
Class-wise correct: Class 0 = 31669.0, Class 1 = 8374.0, Class 2 = 15202.0
Precision: 69.47  Recall: 70.30  F1: 69.66
Confusion Matrix:
 [[31669  1362  3532]
 [ 1711  8374  3087]
 [ 4209  5628 15202]]
VAL Loss after epoch 0: 0.7490369997452085 VAL ACC after epoch 0: 0.7388263300077567
Test Loss after epoch 0: 0.7490369997452085 Test Acc after epoch 0: 0.7388263300077567
WEIGHTS SAVED


Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x704f348de160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Training: 100%|██████████| 4531/4531 [10:10<00:00,  7.42it/s]


...

Correct classified: 55630.0 / 74774
Class-wise correct: Class 0 = 32842.0, Class 1 = 6140.0, Class 2 = 16648.0
Precision: 67.91  Recall: 67.64  F1: 67.73
Confusion Matrix:
 [[32842  1107  2614]
 [ 3052  6140  3980]
 [ 2373  6018 16648]]
VAL Loss after epoch 28: 0.7037711433099176 VAL ACC after epoch 28: 0.7439751785379944
Test Loss after epoch 28: 0.7037711433099176 Test Acc after epoch 28: 0.7439751785379944


Training:   0%|          | 0/4531 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x704f348de160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Training: 100%|██████████| 4531/4531 [10:07<00:00,  7.46it/s]


Correct Classified: 229061.0 / 289972
Class-wise correct: Class 0 = 123474.0, Class 1 = 39263.0, Class 2 = 66324.0
Train Loss after epoch 31: 1.480479433741219 Train ACC after epoch 29: 0.7899417874829294


Eval:   0%|          | 0/74774 [00:00<?, ?it/s]Exception ignored in: <function tqdm.__del__ at 0x704f348de160>
Traceback (most recent call last):
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/anaconda/anaconda3/envs/braindecode/lib/python3.8/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
/media/dll-1/SSD 4TB/code/eeg_classifiers/EEDDataLoader.py:59: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  wavelet_transform = torch.tensor(coefficients_cmor, dtype=torch.float32)
Eval: 100%|██████████| 74774/74774 [03:42<00:00, 335.51it/s]


Correct classified: 55768.0 / 74774
Class-wise correct: Class 0 = 32804.0, Class 1 = 6454.0, Class 2 = 16510.0
Precision: 68.32  Recall: 68.22  F1: 68.21
Confusion Matrix:
 [[32804  1179  2580]
 [ 2812  6454  3906]
 [ 2367  6162 16510]]
VAL Loss after epoch 29: 0.7057828069949637 VAL ACC after epoch 29: 0.7458207398293525
Test Loss after epoch 29: 0.7057828069949637 Test Acc after epoch 29: 0.7458207398293525


In [10]:
print("Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  ")
for epoch in range(epochs):
    if(saved_epoch == epoch):
        print("Saved Weights")
    print(result['epoch'][epoch],"|",result['train_loss'][epoch],"|",result['train_acc'][epoch],"|",result['eval_loss'][epoch],"|",result['eval_acc'][epoch],"|",result['test_loss'][epoch],"|",result['test_acc'][epoch])

Epoch | Train Loss | Train Acc | Eval Loss | Eval Acc | Test Loss | Test Acc  
0 | 1.7115727017136815 | 0.7073096712786062 | 0.7490369997452085 | 0.7388263300077567 | 0.7490369997452085 | 0.7388263300077567
1 | 1.565009873846303 | 0.7542693777330225 | 0.7180187308783038 | 0.7499264450209966 | 0.7180187308783038 | 0.7499264450209966
2 | 1.5363902926734414 | 0.7663981349923441 | 0.709407098738199 | 0.749070532538048 | 0.709407098738199 | 0.749070532538048
3 | 1.520856993689334 | 0.7725332101030444 | 0.6986910797111342 | 0.7524406879396581 | 0.6986910797111342 | 0.7524406879396581
4 | 1.512639044715563 | 0.7766198115680135 | 0.6946371693292548 | 0.7523069516141975 | 0.6946371693292548 | 0.7523069516141975
5 | 1.5059147020018777 | 0.7794683624625826 | 0.7045265412119069 | 0.7543129964961083 | 0.7045265412119069 | 0.7543129964961083
6 | 1.5013228240746637 | 0.7805201881560978 | 0.7000474254803349 | 0.7548613154304973 | 0.7000474254803349 | 0.7548613154304973
Saved Weights
7 | 1.498149831730